In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STAGE 1: LOAD PREPROCESSED DATA
# ============================================================
print("="*60)
print("STAGE 1: LOADING PREPROCESSED DATA")
print("="*60)

batsman_df = pd.read_csv('/kaggle/input/cricket-ds-final/batsman_match_stats.csv')
bowler_df = pd.read_csv('/kaggle/input/cricket-ds-final/bowler_match_stats.csv')


batsman_df['date'] = pd.to_datetime(batsman_df['date'])
bowler_df['date'] = pd.to_datetime(bowler_df['date'])

batsman_df = batsman_df.sort_values(['player', 'date']).reset_index(drop=True)
bowler_df = bowler_df.sort_values(['player', 'date']).reset_index(drop=True)

print(f"✅ Batsman data: {batsman_df.shape}")
print(f"✅ Bowler data: {bowler_df.shape}")
print(f"✅ Date range: {batsman_df['date'].min()} to {batsman_df['date'].max()}")

STAGE 1: LOADING PREPROCESSED DATA
✅ Batsman data: (14805, 20)
✅ Bowler data: (11542, 20)
✅ Date range: 2008-04-18 00:00:00 to 2025-06-03 00:00:00


In [2]:
# ============================================================
# STAGE 2: ROLLING AVERAGES (RECENT FORM FEATURES)
# ============================================================
print("\n" + "="*60)
print("STAGE 2: CREATING ROLLING AVERAGE FEATURES (FORM)")
print("="*60)

def create_rolling_features(df, player_col, stat_cols, windows=[3, 5, 10]):
    """
    Create rolling average features for recent form
    """
    df_copy = df.copy()
    
    for stat in stat_cols:
        for window in windows:
            col_name = f'{stat}_last_{window}'
            df_copy[col_name] = df_copy.groupby(player_col)[stat].transform(
                lambda x: x.shift(1).rolling(window=window, min_periods=1).mean()
            )
    
    return df_copy

print("\n📊 Creating batsman rolling features...")
batsman_rolling_stats = ['runs', 'balls_faced', 'strike_rate', 'boundaries']
batsman_df = create_rolling_features(batsman_df, 'player', batsman_rolling_stats, windows=[3, 5, 10])

print("Created features:")
for stat in batsman_rolling_stats:
    print(f"  - {stat}_last_3, {stat}_last_5, {stat}_last_10")

print("\n📊 Creating bowler rolling features...")
bowler_rolling_stats = ['wickets', 'runs_conceded', 'economy', 'balls_bowled']
bowler_df = create_rolling_features(bowler_df, 'player', bowler_rolling_stats, windows=[3, 5, 10])

print("Created features:")
for stat in bowler_rolling_stats:
    print(f"  - {stat}_last_3, {stat}_last_5, {stat}_last_10")

print(f"\n✅ Batsman features after rolling: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after rolling: {bowler_df.shape[1]} columns")


STAGE 2: CREATING ROLLING AVERAGE FEATURES (FORM)

📊 Creating batsman rolling features...
Created features:
  - runs_last_3, runs_last_5, runs_last_10
  - balls_faced_last_3, balls_faced_last_5, balls_faced_last_10
  - strike_rate_last_3, strike_rate_last_5, strike_rate_last_10
  - boundaries_last_3, boundaries_last_5, boundaries_last_10

📊 Creating bowler rolling features...
Created features:
  - wickets_last_3, wickets_last_5, wickets_last_10
  - runs_conceded_last_3, runs_conceded_last_5, runs_conceded_last_10
  - economy_last_3, economy_last_5, economy_last_10
  - balls_bowled_last_3, balls_bowled_last_5, balls_bowled_last_10

✅ Batsman features after rolling: 32 columns
✅ Bowler features after rolling: 32 columns


In [3]:

# ============================================================
# STAGE 3: CAREER STATISTICS (CUMULATIVE FEATURES)
# ============================================================
print("\n" + "="*60)
print("STAGE 3: CREATING CAREER STATISTICS")
print("="*60)

print("\n📊 Creating batsman career features...")

batsman_df['career_runs'] = batsman_df.groupby('player')['runs'].transform(
    lambda x: x.shift(1).expanding().sum()
)
batsman_df['career_matches'] = batsman_df.groupby('player').cumcount()
batsman_df['career_avg_runs'] = (
    batsman_df['career_runs'] / batsman_df['career_matches']
).fillna(0)

batsman_df['career_total_balls'] = batsman_df.groupby('player')['balls_faced'].transform(
    lambda x: x.shift(1).expanding().sum()
)
batsman_df['career_strike_rate'] = (
    batsman_df['career_runs'] / batsman_df['career_total_balls'] * 100
).fillna(0)

batsman_df['career_boundaries'] = batsman_df.groupby('player')['boundaries'].transform(
    lambda x: x.shift(1).expanding().sum()
)
batsman_df['career_boundary_percentage'] = (
    batsman_df['career_boundaries'] / batsman_df['career_total_balls'] * 100
).fillna(0)

batsman_df['career_dismissals'] = batsman_df.groupby('player')['got_out'].transform(
    lambda x: x.shift(1).expanding().sum()
)

print("Created career features:")
print("  - career_runs, career_matches, career_avg_runs")
print("  - career_strike_rate, career_boundaries, career_boundary_percentage")
print("  - career_dismissals")

# BOWLER CAREER STATS
print("\n📊 Creating bowler career features...")

# Cumulative career stats
bowler_df['career_wickets'] = bowler_df.groupby('player')['wickets'].transform(
    lambda x: x.shift(1).expanding().sum()
)
bowler_df['career_matches'] = bowler_df.groupby('player').cumcount()
bowler_df['career_avg_wickets'] = (
    bowler_df['career_wickets'] / bowler_df['career_matches']
).fillna(0)

bowler_df['career_runs_conceded'] = bowler_df.groupby('player')['runs_conceded'].transform(
    lambda x: x.shift(1).expanding().sum()
)
bowler_df['career_balls_bowled'] = bowler_df.groupby('player')['balls_bowled'].transform(
    lambda x: x.shift(1).expanding().sum()
)
bowler_df['career_economy'] = (
    bowler_df['career_runs_conceded'] / bowler_df['career_balls_bowled'].replace(0, np.nan) * 6
).fillna(0)

bowler_df['career_bowling_avg'] = (
    bowler_df['career_runs_conceded'] / bowler_df['career_wickets'].replace(0, np.nan)
).fillna(0)

print("Created career features:")
print("  - career_wickets, career_matches, career_avg_wickets")
print("  - career_economy, career_bowling_avg")

print(f"\n✅ Batsman features after career stats: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after career stats: {bowler_df.shape[1]} columns")


STAGE 3: CREATING CAREER STATISTICS

📊 Creating batsman career features...
Created career features:
  - career_runs, career_matches, career_avg_runs
  - career_strike_rate, career_boundaries, career_boundary_percentage
  - career_dismissals

📊 Creating bowler career features...
Created career features:
  - career_wickets, career_matches, career_avg_wickets
  - career_economy, career_bowling_avg

✅ Batsman features after career stats: 40 columns
✅ Bowler features after career stats: 39 columns


In [4]:
# ============================================================
# STAGE 4: VENUE-SPECIFIC STATISTICS
# ============================================================
print("\n" + "="*60)
print("STAGE 4: CREATING VENUE-SPECIFIC FEATURES")
print("="*60)

print("\n📊 Creating batsman venue features...")

venue_batting_stats = batsman_df.groupby(['player', 'venue']).apply(
    lambda x: x.sort_values('date').assign(
        venue_runs_avg=lambda df: df['runs'].shift(1).expanding().mean(),
        venue_strike_rate_avg=lambda df: df['strike_rate'].shift(1).expanding().mean(),
        venue_matches_played=lambda df: df['runs'].shift(1).expanding().count()
    )
).reset_index(drop=True)

batsman_df['venue_runs_avg'] = venue_batting_stats['venue_runs_avg'].fillna(0)
batsman_df['venue_strike_rate_avg'] = venue_batting_stats['venue_strike_rate_avg'].fillna(0)
batsman_df['venue_matches_played'] = venue_batting_stats['venue_matches_played'].fillna(0)

batsman_df['venue_familiar'] = (batsman_df['venue_matches_played'] >= 3).astype(int)

print("Created venue features:")
print("  - venue_runs_avg, venue_strike_rate_avg")
print("  - venue_matches_played, venue_familiar")

print("\n📊 Creating bowler venue features...")

venue_bowling_stats = bowler_df.groupby(['player', 'venue']).apply(
    lambda x: x.sort_values('date').assign(
        venue_wickets_avg=lambda df: df['wickets'].shift(1).expanding().mean(),
        venue_economy_avg=lambda df: df['economy'].shift(1).expanding().mean(),
        venue_matches_played=lambda df: df['wickets'].shift(1).expanding().count()
    )
).reset_index(drop=True)

bowler_df['venue_wickets_avg'] = venue_bowling_stats['venue_wickets_avg'].fillna(0)
bowler_df['venue_economy_avg'] = venue_bowling_stats['venue_economy_avg'].fillna(0)
bowler_df['venue_matches_played'] = venue_bowling_stats['venue_matches_played'].fillna(0)
bowler_df['venue_familiar'] = (bowler_df['venue_matches_played'] >= 3).astype(int)

print("Created venue features:")
print("  - venue_wickets_avg, venue_economy_avg")
print("  - venue_matches_played, venue_familiar")

print(f"\n✅ Batsman features after venue stats: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after venue stats: {bowler_df.shape[1]} columns")




STAGE 4: CREATING VENUE-SPECIFIC FEATURES

📊 Creating batsman venue features...
Created venue features:
  - venue_runs_avg, venue_strike_rate_avg
  - venue_matches_played, venue_familiar

📊 Creating bowler venue features...
Created venue features:
  - venue_wickets_avg, venue_economy_avg
  - venue_matches_played, venue_familiar

✅ Batsman features after venue stats: 44 columns
✅ Bowler features after venue stats: 43 columns


In [5]:
# ============================================================
# STAGE 5: OPPONENT-SPECIFIC STATISTICS
# ============================================================
print("\n" + "="*60)
print("STAGE 5: CREATING OPPONENT-SPECIFIC FEATURES")
print("="*60)

print("\n📊 Creating batsman vs opponent features...")

opponent_batting_stats = batsman_df.groupby(['player', 'opponent']).apply(
    lambda x: x.sort_values('date').assign(
        vs_opponent_runs_avg=lambda df: df['runs'].shift(1).expanding().mean(),
        vs_opponent_strike_rate_avg=lambda df: df['strike_rate'].shift(1).expanding().mean(),
        vs_opponent_matches=lambda df: df['runs'].shift(1).expanding().count()
    )
).reset_index(drop=True)

batsman_df['vs_opponent_runs_avg'] = opponent_batting_stats['vs_opponent_runs_avg'].fillna(0)
batsman_df['vs_opponent_strike_rate_avg'] = opponent_batting_stats['vs_opponent_strike_rate_avg'].fillna(0)
batsman_df['vs_opponent_matches'] = opponent_batting_stats['vs_opponent_matches'].fillna(0)

print("Created opponent features:")
print("  - vs_opponent_runs_avg, vs_opponent_strike_rate_avg")
print("  - vs_opponent_matches")

print("\n📊 Creating bowler vs opponent features...")

opponent_bowling_stats = bowler_df.groupby(['player', 'opponent']).apply(
    lambda x: x.sort_values('date').assign(
        vs_opponent_wickets_avg=lambda df: df['wickets'].shift(1).expanding().mean(),
        vs_opponent_economy_avg=lambda df: df['economy'].shift(1).expanding().mean(),
        vs_opponent_matches=lambda df: df['wickets'].shift(1).expanding().count()
    )
).reset_index(drop=True)

bowler_df['vs_opponent_wickets_avg'] = opponent_bowling_stats['vs_opponent_wickets_avg'].fillna(0)
bowler_df['vs_opponent_economy_avg'] = opponent_bowling_stats['vs_opponent_economy_avg'].fillna(0)
bowler_df['vs_opponent_matches'] = opponent_bowling_stats['vs_opponent_matches'].fillna(0)

print("Created opponent features:")
print("  - vs_opponent_wickets_avg, vs_opponent_economy_avg")
print("  - vs_opponent_matches")

print(f"\n✅ Batsman features after opponent stats: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after opponent stats: {bowler_df.shape[1]} columns")



STAGE 5: CREATING OPPONENT-SPECIFIC FEATURES

📊 Creating batsman vs opponent features...
Created opponent features:
  - vs_opponent_runs_avg, vs_opponent_strike_rate_avg
  - vs_opponent_matches

📊 Creating bowler vs opponent features...
Created opponent features:
  - vs_opponent_wickets_avg, vs_opponent_economy_avg
  - vs_opponent_matches

✅ Batsman features after opponent stats: 47 columns
✅ Bowler features after opponent stats: 46 columns


In [6]:
# ============================================================
# STAGE 6: CONTEXTUAL FEATURES
# ============================================================
print("\n" + "="*60)
print("STAGE 6: CREATING CONTEXTUAL FEATURES")
print("="*60)

print("\n📊 Creating batsman contextual features...")

team_city_map = {
    'Mumbai Indians': 'Mumbai',
    'Chennai Super Kings': 'Chennai',
    'Kolkata Knight Riders': 'Kolkata',
    'Royal Challengers Bangalore': 'Bangalore',
    'Royal Challengers Bengaluru': 'Bengaluru',
    'Delhi Capitals': 'Delhi',
    'Delhi Daredevils': 'Delhi',
    'Sunrisers Hyderabad': 'Hyderabad',
    'Rajasthan Royals': 'Jaipur',
    'Punjab Kings': 'Mohali',
    'Kings XI Punjab': 'Mohali',
    'Gujarat Titans': 'Ahmedabad',
    'Lucknow Super Giants': 'Lucknow',
    'Gujarat Lions': 'Rajkot',
    'Rising Pune Supergiant': 'Pune',
    'Pune Warriors': 'Pune',
    'Deccan Chargers': 'Hyderabad',
    'Kochi Tuskers Kerala': 'Kochi'
}

batsman_df['team_home_city'] = batsman_df['team'].map(team_city_map)

batsman_df['is_home_match'] = (
    batsman_df['city'].fillna('').str.lower() == 
    batsman_df['team_home_city'].fillna('').str.lower()
).astype(int)

batsman_df = batsman_df.drop('team_home_city', axis=1)

batsman_df['days_since_last_match'] = batsman_df.groupby('player')['date'].diff().dt.days.fillna(0)

batsman_df['days_since_last_match'] = batsman_df['days_since_last_match'].clip(0, 90)

batsman_df['season_matches'] = batsman_df.groupby(['player', 'season']).cumcount()

batsman_df['form_indicator'] = (
    batsman_df['runs_last_5'] - batsman_df['career_avg_runs']
).fillna(0)

batsman_df['performance_trend'] = (
    batsman_df['runs_last_3'] - batsman_df['runs_last_10']
).fillna(0)

print("Created contextual features:")
print("  - is_home_match, days_since_last_match")
print("  - season_matches, form_indicator")
print("  - performance_trend")

print("\n📊 Creating bowler contextual features...")

bowler_df['team_home_city'] = bowler_df['team'].map(team_city_map)

bowler_df['is_home_match'] = (
    bowler_df['city'].fillna('').str.lower() == 
    bowler_df['team_home_city'].fillna('').str.lower()
).astype(int)

bowler_df = bowler_df.drop('team_home_city', axis=1)

bowler_df['days_since_last_match'] = bowler_df.groupby('player')['date'].diff().dt.days.fillna(0)
bowler_df['days_since_last_match'] = bowler_df['days_since_last_match'].clip(0, 90)

bowler_df['season_matches'] = bowler_df.groupby(['player', 'season']).cumcount()

bowler_df['form_indicator'] = (
    bowler_df['wickets_last_5'] - bowler_df['career_avg_wickets']
).fillna(0)

bowler_df['performance_trend'] = (
    bowler_df['wickets_last_3'] - bowler_df['wickets_last_10']
).fillna(0)

print("Created contextual features:")
print("  - is_home_match, days_since_last_match")
print("  - season_matches, form_indicator")
print("  - performance_trend")

print(f"\n✅ Batsman features after contextual: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after contextual: {bowler_df.shape[1]} columns")



STAGE 6: CREATING CONTEXTUAL FEATURES

📊 Creating batsman contextual features...
Created contextual features:
  - is_home_match, days_since_last_match
  - season_matches, form_indicator
  - performance_trend

📊 Creating bowler contextual features...
Created contextual features:
  - is_home_match, days_since_last_match
  - season_matches, form_indicator
  - performance_trend

✅ Batsman features after contextual: 52 columns
✅ Bowler features after contextual: 51 columns


In [7]:
# ============================================================
# STAGE 7: ADVANCED PERFORMANCE METRICS
# ============================================================
print("\n" + "="*60)
print("STAGE 7: CREATING ADVANCED PERFORMANCE METRICS")
print("="*60)

print("\n📊 Creating batsman advanced metrics...")

batsman_df['runs_std_last_10'] = batsman_df.groupby('player')['runs'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=3).std()
).fillna(0)

batsman_df['consistency_score'] = np.where(
    batsman_df['runs_last_10'] > 0,
    1 - (batsman_df['runs_std_last_10'] / batsman_df['runs_last_10']),
    0
).clip(0, 1)

batsman_df['boundary_dependency'] = np.where(
    batsman_df['runs'] > 0,
    (batsman_df['boundaries'] * 4) / batsman_df['runs'],  # Simplified
    0
).clip(0, 1)

batsman_df['momentum_score'] = (
    0.5 * batsman_df['runs_last_3'] +
    0.3 * batsman_df['runs_last_5'] +
    0.2 * batsman_df['runs_last_10']
).fillna(0)

batsman_df['experience_level'] = pd.cut(
    batsman_df['career_matches'],
    bins=[0, 20, 50, 100, 500],
    labels=['rookie', 'intermediate', 'experienced', 'veteran']
).astype(str)

print("Created advanced metrics:")
print("  - consistency_score, boundary_dependency")
print("  - momentum_score, experience_level")

# BOWLER ADVANCED Metrics
print("\n📊 Creating bowler advanced metrics...")

bowler_df['wickets_std_last_10'] = bowler_df.groupby('player')['wickets'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=3).std()
).fillna(0)

bowler_df['consistency_score'] = np.where(
    bowler_df['wickets_last_10'] > 0,
    1 - (bowler_df['wickets_std_last_10'] / bowler_df['wickets_last_10']),
    0
).clip(0, 1)

bowler_df['wicket_taking_rate'] = np.where(
    bowler_df['overs_bowled'] > 0,
    bowler_df['wickets'] / bowler_df['overs_bowled'],
    0
)

bowler_df['momentum_score'] = (
    0.5 * bowler_df['wickets_last_3'] +
    0.3 * bowler_df['wickets_last_5'] +
    0.2 * bowler_df['wickets_last_10']
).fillna(0)

bowler_df['experience_level'] = pd.cut(
    bowler_df['career_matches'],
    bins=[0, 20, 50, 100, 500],
    labels=['rookie', 'intermediate', 'experienced', 'veteran']
).astype(str)

print("Created advanced metrics:")
print("  - consistency_score, wicket_taking_rate")
print("  - momentum_score, experience_level")

print(f"\n✅ Batsman features after advanced metrics: {batsman_df.shape[1]} columns")
print(f"✅ Bowler features after advanced metrics: {bowler_df.shape[1]} columns")


STAGE 7: CREATING ADVANCED PERFORMANCE METRICS

📊 Creating batsman advanced metrics...
Created advanced metrics:
  - consistency_score, boundary_dependency
  - momentum_score, experience_level

📊 Creating bowler advanced metrics...
Created advanced metrics:
  - consistency_score, wicket_taking_rate
  - momentum_score, experience_level

✅ Batsman features after advanced metrics: 57 columns
✅ Bowler features after advanced metrics: 56 columns


In [8]:
# ============================================================
# STAGE 8: CLEAN AND PREPARE FINAL DATASET
# ============================================================
print("\n" + "="*60)
print("STAGE 8: CLEANING AND PREPARING FINAL DATASET")
print("="*60)

print("\n🧹 Cleaning infinite values...")
batsman_df = batsman_df.replace([np.inf, -np.inf], np.nan)
bowler_df = bowler_df.replace([np.inf, -np.inf], np.nan)

numeric_cols_bat = batsman_df.select_dtypes(include=[np.number]).columns
batsman_df[numeric_cols_bat] = batsman_df[numeric_cols_bat].fillna(0)

numeric_cols_bowl = bowler_df.select_dtypes(include=[np.number]).columns
bowler_df[numeric_cols_bowl] = bowler_df[numeric_cols_bowl].fillna(0)

print("✅ Cleaned infinite and NaN values")

print("\n🎯 Filtering rows with valid targets...")
batsman_final = batsman_df[batsman_df['target_runs'].notna()].copy()
bowler_final = bowler_df[bowler_df['target_wickets'].notna()].copy()

print(f"Batsman: {len(batsman_df)} → {len(batsman_final)} rows (removed {len(batsman_df) - len(batsman_final)} without targets)")
print(f"Bowler: {len(bowler_df)} → {len(bowler_final)} rows (removed {len(bowler_df) - len(bowler_final)} without targets)")




STAGE 8: CLEANING AND PREPARING FINAL DATASET

🧹 Cleaning infinite values...
✅ Cleaned infinite and NaN values

🎯 Filtering rows with valid targets...
Batsman: 14805 → 14805 rows (removed 0 without targets)
Bowler: 11542 → 11542 rows (removed 0 without targets)


In [9]:
# ============================================================
# STAGE 9: FEATURE SUMMARY AND VALIDATION
# ============================================================
print("\n" + "="*60)
print("STAGE 9: FEATURE ENGINEERING SUMMARY")
print("="*60)

print("\n📊 BATSMAN FEATURES SUMMARY:")
print(f"   Total features: {batsman_final.shape[1]}")
print(f"   Total records: {len(batsman_final):,}")
print(f"   Players: {batsman_final['player'].nunique()}")
print(f"   Date range: {batsman_final['date'].min()} to {batsman_final['date'].max()}")

print("\n📋 Feature Categories:")
print(f"   - Original features: 20")
print(f"   - Rolling averages (3,5,10): {len([c for c in batsman_final.columns if 'last_' in c])}")
print(f"   - Career statistics: {len([c for c in batsman_final.columns if 'career_' in c])}")
print(f"   - Venue-specific: {len([c for c in batsman_final.columns if 'venue_' in c])}")
print(f"   - Opponent-specific: {len([c for c in batsman_final.columns if 'vs_opponent_' in c or 'opponent_' in c])}")
print(f"   - Contextual: {len([c for c in batsman_final.columns if c in ['is_home_match', 'days_since_last_match', 'season_matches', 'form_indicator']])}")
print(f"   - Advanced metrics: {len([c for c in batsman_final.columns if c in ['consistency_score', 'boundary_dependency', 'momentum_score', 'experience_level']])}")

print("\n📊 BOWLER FEATURES SUMMARY:")
print(f"   Total features: {bowler_final.shape[1]}")
print(f"   Total records: {len(bowler_final):,}")
print(f"   Players: {bowler_final['player'].nunique()}")
print(f"   Date range: {bowler_final['date'].min()} to {bowler_final['date'].max()}")

print("\n📋 Feature Categories:")
print(f"   - Original features: 20")
print(f"   - Rolling averages (3,5,10): {len([c for c in bowler_final.columns if 'last_' in c])}")
print(f"   - Career statistics: {len([c for c in bowler_final.columns if 'career_' in c])}")
print(f"   - Venue-specific: {len([c for c in bowler_final.columns if 'venue_' in c])}")
print(f"   - Opponent-specific: {len([c for c in bowler_final.columns if 'vs_opponent_' in c or 'opponent_' in c])}")
print(f"   - Contextual: {len([c for c in bowler_final.columns if c in ['is_home_match', 'days_since_last_match', 'season_matches', 'form_indicator']])}")
print(f"   - Advanced metrics: {len([c for c in bowler_final.columns if c in ['consistency_score', 'wicket_taking_rate', 'momentum_score', 'experience_level']])}")

print("\n📄 Sample Batsman Features (first 3 rows):")
feature_cols = ['player', 'date', 'runs', 'runs_last_5', 'career_avg_runs', 
                'venue_runs_avg', 'vs_opponent_runs_avg', 'momentum_score', 'target_runs']
print(batsman_final[feature_cols].head(3))

print("\n📄 Sample Bowler Features (first 3 rows):")
feature_cols = ['player', 'date', 'wickets', 'wickets_last_5', 'career_avg_wickets',
                'venue_wickets_avg', 'vs_opponent_wickets_avg', 'momentum_score', 'target_wickets']
print(bowler_final[feature_cols].head(3))


STAGE 9: FEATURE ENGINEERING SUMMARY

📊 BATSMAN FEATURES SUMMARY:
   Total features: 57
   Total records: 14,805
   Players: 226
   Date range: 2008-04-18 00:00:00 to 2025-06-03 00:00:00

📋 Feature Categories:
   - Original features: 20
   - Rolling averages (3,5,10): 14
   - Career statistics: 8
   - Venue-specific: 4
   - Opponent-specific: 3
   - Contextual: 4
   - Advanced metrics: 4

📊 BOWLER FEATURES SUMMARY:
   Total features: 56
   Total records: 11,542
   Players: 190
   Date range: 2008-04-18 00:00:00 to 2025-06-03 00:00:00

📋 Feature Categories:
   - Original features: 20
   - Rolling averages (3,5,10): 14
   - Career statistics: 7
   - Venue-specific: 4
   - Opponent-specific: 3
   - Contextual: 4
   - Advanced metrics: 4

📄 Sample Batsman Features (first 3 rows):
           player       date  runs  runs_last_5  career_avg_runs  \
0  A Ashish Reddy 2012-04-29    10          0.0              0.0   
1  A Ashish Reddy 2012-05-04     3         10.0             10.0   
2  A Ash

In [10]:
# ============================================================
# STAGE 10: SAVE ENGINEERED FEATURES
# ============================================================
print("\n" + "="*60)
print("STAGE 10: SAVING ENGINEERED FEATURES")
print("="*60)

# Save to CSV
batsman_final.to_csv('batsman_features_engineered.csv', index=False)
bowler_final.to_csv('bowler_features_engineered.csv', index=False)

print("\n✅ Saved files:")
print("   - batsman_features_engineered.csv")
print("   - bowler_features_engineered.csv")

batsman_feature_list = [col for col in batsman_final.columns 
                        if col not in ['match_id', 'date', 'player', 'team', 'opponent', 
                                       'venue', 'city', 'season', 'innings',
                                       'target_runs', 'target_balls_faced', 'target_strike_rate',
                                       'next_opponent', 'next_venue']]

bowler_feature_list = [col for col in bowler_final.columns 
                       if col not in ['match_id', 'date', 'player', 'team', 'opponent',
                                      'venue', 'city', 'season', 'innings',
                                      'target_wickets', 'target_runs_conceded', 'target_economy',
                                      'next_opponent', 'next_venue']]

pd.DataFrame({'batsman_features': batsman_feature_list}).to_csv('batsman_feature_list.csv', index=False)
pd.DataFrame({'bowler_features': bowler_feature_list}).to_csv('bowler_feature_list.csv', index=False)

print("\n✅ Saved feature lists:")
print("   - batsman_feature_list.csv")
print("   - bowler_feature_list.csv")

print(f"\n📊 Batsman modeling features: {len(batsman_feature_list)}")
print(f"📊 Bowler modeling features: {len(bowler_feature_list)}")



STAGE 10: SAVING ENGINEERED FEATURES

✅ Saved files:
   - batsman_features_engineered.csv
   - bowler_features_engineered.csv

✅ Saved feature lists:
   - batsman_feature_list.csv
   - bowler_feature_list.csv

📊 Batsman modeling features: 43
📊 Bowler modeling features: 42


In [11]:
# ============================================================
# STAGE 11: DATA QUALITY CHECKS
# ============================================================
print("\n" + "="*60)
print("STAGE 11: FINAL DATA QUALITY CHECKS")
print("="*60)

print("\n🔍 Batsman Data Quality:")
print(f"   Missing values: {batsman_final.isnull().sum().sum()}")
print(f"   Infinite values: {np.isinf(batsman_final.select_dtypes(include=[np.number])).sum().sum()}")
print(f"   Duplicate rows: {batsman_final.duplicated().sum()}")
print(f"   Target variable distribution:")
print(f"      Mean: {batsman_final['target_runs'].mean():.2f}")
print(f"      Median: {batsman_final['target_runs'].median():.2f}")
print(f"      Std: {batsman_final['target_runs'].std():.2f}")

print("\n🔍 Bowler Data Quality:")
print(f"   Missing values: {bowler_final.isnull().sum().sum()}")
print(f"   Infinite values: {np.isinf(bowler_final.select_dtypes(include=[np.number])).sum().sum()}")
print(f"   Duplicate rows: {bowler_final.duplicated().sum()}")
print(f"   Target variable distribution:")
print(f"      Mean: {bowler_final['target_wickets'].mean():.2f}")
print(f"      Median: {bowler_final['target_wickets'].median():.2f}")
print(f"      Std: {bowler_final['target_wickets'].std():.2f}")

print("\n" + "="*60)
print("✅ FEATURE ENGINEERING COMPLETE!")
print("="*60)

print("\n🎯 READY FOR MODELING!")
print("📁 Generated Files:")
print("   1. batsman_features_engineered.csv")
print("   2. bowler_features_engineered.csv")
print("   3. batsman_feature_list.csv")
print("   4. bowler_feature_list.csv")


STAGE 11: FINAL DATA QUALITY CHECKS

🔍 Batsman Data Quality:
   Missing values: 452
   Infinite values: 0
   Duplicate rows: 0
   Target variable distribution:
      Mean: 21.72
      Median: 15.00
      Std: 22.18

🔍 Bowler Data Quality:
   Missing values: 380
   Infinite values: 0
   Duplicate rows: 0
   Target variable distribution:
      Mean: 1.01
      Median: 1.00
      Std: 1.06

✅ FEATURE ENGINEERING COMPLETE!

🎯 READY FOR MODELING!
📁 Generated Files:
   1. batsman_features_engineered.csv
   2. bowler_features_engineered.csv
   3. batsman_feature_list.csv
   4. bowler_feature_list.csv


In [13]:
import pandas as pd
import numpy as np

batsman_df = pd.read_csv('/kaggle/working/batsman_features_engineered.csv')
bowler_df = pd.read_csv('/kaggle/working/bowler_features_engineered.csv')

print("="*60)
print("MISSING VALUES ANALYSIS")
print("="*60)

print("\n📊 BATSMAN MISSING VALUES:")
missing_bat = batsman_df.isnull().sum()
missing_bat = missing_bat[missing_bat > 0].sort_values(ascending=False)
print(missing_bat)
print(f"\nTotal missing: {batsman_df.isnull().sum().sum()}")
print(f"Percentage: {(batsman_df.isnull().sum().sum() / (batsman_df.shape[0] * batsman_df.shape[1]) * 100):.2f}%")

print("\n📊 BOWLER MISSING VALUES:")
missing_bowl = bowler_df.isnull().sum()
missing_bowl = missing_bowl[missing_bowl > 0].sort_values(ascending=False)
print(missing_bowl)
print(f"\nTotal missing: {bowler_df.isnull().sum().sum()}")
print(f"Percentage: {(bowler_df.isnull().sum().sum() / (bowler_df.shape[0] * bowler_df.shape[1]) * 100):.2f}%")

print("\n🔍 COLUMNS WITH MISSING VALUES:")
print("\nBatsman columns with >5% missing:")
for col in missing_bat.index[:10]:
    pct = (missing_bat[col] / len(batsman_df)) * 100
    print(f"  {col}: {missing_bat[col]} ({pct:.2f}%)")

print("\nBowler columns with >5% missing:")
for col in missing_bowl.index[:10]:
    pct = (missing_bowl[col] / len(bowler_df)) * 100
    print(f"  {col}: {missing_bowl[col]} ({pct:.2f}%)")


MISSING VALUES ANALYSIS

📊 BATSMAN MISSING VALUES:
next_opponent       226
next_venue          226
experience_level    226
dtype: int64

Total missing: 678
Percentage: 0.08%

📊 BOWLER MISSING VALUES:
next_opponent       190
next_venue          190
experience_level    190
dtype: int64

Total missing: 570
Percentage: 0.09%

🔍 COLUMNS WITH MISSING VALUES:

Batsman columns with >5% missing:
  next_opponent: 226 (1.53%)
  next_venue: 226 (1.53%)
  experience_level: 226 (1.53%)

Bowler columns with >5% missing:
  next_opponent: 190 (1.65%)
  next_venue: 190 (1.65%)
  experience_level: 190 (1.65%)


In [14]:
import pandas as pd
import numpy as np

print("="*60)
print("FIXING MISSING VALUES")
print("="*60)

batsman_df = pd.read_csv('/kaggle/working/batsman_features_engineered.csv')
bowler_df = pd.read_csv('/kaggle/working/bowler_features_engineered.csv')

print(f"\n📊 Before fix:")
print(f"Batsman missing: {batsman_df.isnull().sum().sum()}")
print(f"Bowler missing: {bowler_df.isnull().sum().sum()}")

batsman_df['next_opponent'] = batsman_df['next_opponent'].fillna('unknown')
batsman_df['next_venue'] = batsman_df['next_venue'].fillna('unknown')
batsman_df['experience_level'] = batsman_df['experience_level'].fillna('rookie')

bowler_df['next_opponent'] = bowler_df['next_opponent'].fillna('unknown')
bowler_df['next_venue'] = bowler_df['next_venue'].fillna('unknown')
bowler_df['experience_level'] = bowler_df['experience_level'].fillna('rookie')

print(f"\n📊 After fix:")
print(f"Batsman missing: {batsman_df.isnull().sum().sum()}")
print(f"Bowler missing: {bowler_df.isnull().sum().sum()}")

batsman_df.to_csv('batsman_features_final.csv', index=False)
bowler_df.to_csv('bowler_features_final.csv', index=False)

print("\n✅ Saved final files:")
print("   - batsman_features_final.csv")
print("   - bowler_features_final.csv")

assert batsman_df.isnull().sum().sum() == 0, "Still have missing values!"
assert bowler_df.isnull().sum().sum() == 0, "Still have missing values!"

print("\n" + "="*60)
print("✅ DATA READY FOR MODELING!")
print("="*60)


FIXING MISSING VALUES

📊 Before fix:
Batsman missing: 678
Bowler missing: 570

📊 After fix:
Batsman missing: 0
Bowler missing: 0

✅ Saved final files:
   - batsman_features_final.csv
   - bowler_features_final.csv

✅ DATA READY FOR MODELING!


In [15]:
# ==============================================================================
# STAGE 12: CREATE & SAVE PREPROCESSING PIPELINES (FINAL DELIVERABLE)
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

print("="*60)
print("STAGE 12: CREATING & SAVING PREPROCESSING PIPELINES")
print("="*60)

cat_cols = ['team', 'opponent', 'venue', 'city', 'innings', 'experience_level']

metadata_cols = ['match_id', 'player', 'date', 'season']
future_cols = ['next_opponent', 'next_venue']

batsman_targets = ['target_runs', 'target_balls_faced', 'target_strike_rate']
bowler_targets = ['target_wickets', 'target_runs_conceded', 'target_economy']

print("\n🏏 Building Batsman Pipeline...")
exclude_bat = set(cat_cols + metadata_cols + future_cols + batsman_targets)
# Use 'batsman_df' from your previous cell
num_cols_bat = [c for c in batsman_df.columns if c not in exclude_bat]

print(f"   Categorical ({len(cat_cols)}): {cat_cols}")
print(f"   Numeric ({len(num_cols_bat)}): {num_cols_bat[:5]} ... (Total: {len(num_cols_bat)})")

bat_pipeline = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_bat),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

print("\n🥎 Building Bowler Pipeline...")
exclude_bowl = set(cat_cols + metadata_cols + future_cols + bowler_targets)
num_cols_bowl = [c for c in bowler_df.columns if c not in exclude_bowl]

print(f"   Categorical ({len(cat_cols)}): {cat_cols}")
print(f"   Numeric ({len(num_cols_bowl)}): {num_cols_bowl[:5]} ... (Total: {len(num_cols_bowl)})")

bowl_pipeline = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_bowl),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

bat_pipeline.fit(batsman_df)
bowl_pipeline.fit(bowler_df)

joblib.dump(bat_pipeline, 'feature_pipeline_batsman.pkl')
joblib.dump(bowl_pipeline, 'feature_pipeline_bowler.pkl')

print("\n✅ SUCCESS: Preprocessing pipelines saved!")
print("   - feature_pipeline_batsman.pkl")
print("   - feature_pipeline_bowler.pkl")

STAGE 12: CREATING & SAVING PREPROCESSING PIPELINES

🏏 Building Batsman Pipeline...
   Categorical (6): ['team', 'opponent', 'venue', 'city', 'innings', 'experience_level']
   Numeric (42): ['runs', 'balls_faced', 'boundaries', 'got_out', 'strike_rate'] ... (Total: 42)

🥎 Building Bowler Pipeline...
   Categorical (6): ['team', 'opponent', 'venue', 'city', 'innings', 'experience_level']
   Numeric (41): ['runs_conceded', 'balls_bowled', 'wickets', 'economy', 'overs_bowled'] ... (Total: 41)

✅ SUCCESS: Preprocessing pipelines saved!
   - feature_pipeline_batsman.pkl
   - feature_pipeline_bowler.pkl
